# A1、A2、A3 统一评估

先修改同目录的 config.yaml，再从上到下运行。输入为训练前已追加 1000 个复制区间的真实 H5AD 和已生成的合成 H5AD；本流程不训练生成模型。

- A1：按真实类型占比加权的逐区域 1-WD、类型细胞均值 MSE、区域均值 Pearson。
- A2：原区间与复制区间实际生成值的逐对 Pearson；另输出原始区间的 Spearman 分布。
- A3：按真实类型 ANOVA 筛选原始区间、真实参照拟合 PCA、30-NN miLISI 和两种 UMAP。

A1、A3 均使用 Subclass，全部步骤排除末尾 1000 列。指标不截断合成值，也不替换合成数据的复制列。

In [ ]:
from pathlib import Path

# 从仓库根目录或 evaluate 目录启动均可；也可改为自定义配置的绝对路径。
CONFIG_PATH = Path("config.yaml") if Path("config.yaml").is_file() else Path("evaluate/config.yaml")

## 环境与配置

使用 simscDNAm 环境的 Jupyter 内核。所有相对数据路径相对于配置文件；图中的文本用英文，表格和运行说明用中文。

In [ ]:
from contextlib import ExitStack
import importlib.metadata
import json
import sys

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.decomposition import PCA
from threadpoolctl import threadpool_limits
import umap
import yaml

MODULE_DIR = next((p for p in (Path.cwd() / "evaluate", Path.cwd())
                   if (p / "A1A2A3.ipynb").is_file() and (p / "preprocessing.py").is_file()), None)
if MODULE_DIR is None:
    raise FileNotFoundError("请从仓库根目录或 evaluate 目录启动 notebook，并保留同目录模块")
sys.path.insert(0, str(MODULE_DIR.resolve().parent))
from evaluate.preprocessing import (dense_block, read_labels, region_pairs, validate_real_duplicates,
    natural_reference_statistics, select_features, imputed_features, validate_region_axis, sample_rows)
from evaluate.metrics import a1_metrics, paired_pearson, spearman_vector, milisi

if Path(sys.prefix).name.lower() != "simscdnam":
    raise RuntimeError(f"请切换到 simscDNAm 的 Jupyter 内核，当前为 {sys.executable}")
CONFIG_PATH = CONFIG_PATH.expanduser().resolve()
config = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
if not isinstance(config, dict):
    raise ValueError("配置必须是 YAML 映射")
required = {"real_h5ad", "synthetic_h5ad", "output_dir"}
if not required.issubset(config) or not isinstance(config["synthetic_h5ad"], dict) or not config["synthetic_h5ad"]:
    raise ValueError("请填写 real_h5ad、非空 synthetic_h5ad 方法映射和 output_dir")
defaults = {"reference_h5ad": None, "region_pairs_csv": None, "label_key": "Subclass",
    "duplicate_count": 1000, "seed": 42, "evaluation_cells": None, "a3_top_regions": 50000,
    "pca_components": 50, "milisi_neighbors": 30, "umap_seed": 2022, "spearman_top_regions": 2000,
    "a1_regions_csv": None, "genome_assembly": None, "read_in_memory": False}
config = {**defaults, **config}
if not isinstance(config["read_in_memory"], bool):
    raise ValueError("read_in_memory必须为布尔值")
if config["label_key"] != "Subclass":
    raise ValueError("本流程统一使用真实类型 Subclass，请将真实标签字段改名为 Subclass")
if config["duplicate_count"] != 1000:
    raise ValueError("本流程固定最后 1000 个区间为复制区间")
for key in ("a3_top_regions", "pca_components", "milisi_neighbors", "spearman_top_regions"):
    if not isinstance(config[key], int) or isinstance(config[key], bool) or config[key] < 2:
        raise ValueError(f"{key} 必须为不小于 2 的整数")
if config["evaluation_cells"] is not None and (not isinstance(config["evaluation_cells"], int)
        or isinstance(config["evaluation_cells"], bool) or config["evaluation_cells"] < 3):
    raise ValueError("evaluation_cells 必须为 null 或不小于 3 的整数")
if any(not isinstance(name, str) or not name.strip() or name in {"真实数据", "训练参照", "CVAEKMeansAligned"}
       for name in config["synthetic_h5ad"]):
    raise ValueError("方法名须为非空字符串，且不能使用保留名称或暂不参与的 CVAEKMeansAligned")

def resolve_path(value):
    path = Path(value).expanduser()
    return (CONFIG_PATH.parent / path).resolve() if not path.is_absolute() else path.resolve()

plt.rcParams.update({"figure.dpi": 110, "font.family": "DejaVu Sans", "pdf.fonttype": 42,
                     "axes.spines.top": False, "axes.spines.right": False})
print("Jupyter 内核：", sys.executable)
print("配置文件：", CONFIG_PATH)

## 绘图

只输出四类图：文献区域边缘分布、单区域按 Subclass 的边缘分布、A2 复制区间 Pearson 小提琴＋正式均值柱状图、A3 本轮真实／合成联合 UMAP。区域由显式证据清单指定，不按合成结果或留出细胞筛选。

In [ ]:
from matplotlib.colors import to_rgba
from matplotlib.patches import Patch
from matplotlib.ticker import MaxNLocator


PLOT_COLORS = {"真实数据": "#A6A6A6", "Real": "#A6A6A6",
    "simscDNAm": "#3B73B9", "CVAE_training_mask": "#3B73B9", "FittedMarginal": "#269D91",
    "simATAC": "#A487C9", "simCAS": "#D95F02"}


def plot_name(name):
    return {"真实数据": "Real", "CVAE_training_mask": "CVAE (training mask)"}.get(name, name)


def plot_color(name):
    return PLOT_COLORS.get(name, "#3B73B9")


def style_axis(axis):
    axis.spines[["top", "right"]].set_visible(False)
    axis.spines[["left", "bottom"]].set_linewidth(0.7)
    axis.grid(axis="y", color="#E8E8E8", linewidth=0.5)
    axis.set_axisbelow(True)


def save_figure(figure, output, stem):
    with plt.rc_context({"pdf.fonttype": 42}):
        for extension in ("png", "pdf"):
            figure.savefig(output / "figures" / f"{stem}.{extension}", dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(figure)


def probability_histogram(values, edges):
    """按全部有限原值归一化；越界值单独报告，不压入边缘分箱。"""
    finite = np.asarray(values)[np.isfinite(values)]
    probabilities = np.histogram(finite, bins=edges)[0].astype(float)
    outside = np.nan
    if finite.size:
        probabilities /= finite.size
        outside = np.mean((finite < edges[0]) | (finite > edges[-1]))
    return probabilities, outside, int(finite.size)


def load_plot_regions(real, natural_count, path, assembly):
    """显式读取展示证据清单；不把随机区间或复制尾部当作文献区域。"""
    table = pd.read_csv(path)
    required = {"原region", "展示名称", "基因组版本", "文献来源", "单区域展示", "展示细胞类型"}
    if not required.issubset(table) or table.empty:
        raise ValueError("展示区间清单缺少区域、名称、基因组版本、文献来源或单区域展示信息")
    if table[["原region", "展示名称", "基因组版本", "文献来源"]].isna().any().any():
        raise ValueError("展示区间及文献证据不能为空")
    if not assembly or not table["基因组版本"].eq(assembly).all():
        raise ValueError("展示区间清单的基因组版本与配置不一致")
    indices = real.var_names[:natural_count].get_indexer(table["原region"])
    if np.any(indices < 0) or table["原region"].duplicated().any():
        raise ValueError("展示区间须为互不重复且存在的原始区间，不能使用复制尾部")
    flags = pd.to_numeric(table["单区域展示"], errors="coerce")
    if not flags.isin([0, 1]).all() or flags.sum() != 1:
        raise ValueError("清单须且仅须指定一个单区域展示窗口")
    detail = table.loc[flags.eq(1)].iloc[0]
    levels = str(detail["展示细胞类型"]).split("|")
    if pd.isna(detail["展示细胞类型"]) or any(not level.strip() for level in levels) or len(set(levels)) != len(levels):
        raise ValueError("单区域展示须提供以 | 分隔的不重复真实细胞类型")
    table["单区域展示"] = flags.astype(int)
    table.insert(0, "原region索引", indices)
    return table


def marginal_legend(figure, methods, y=0.99):
    handles = [Patch(facecolor=to_rgba(plot_color(name), 0.4), edgecolor=plot_color(name), label=plot_name(name))
               for name in ["真实数据", *methods]]
    figure.legend(handles=handles, loc="upper center", ncol=len(handles), frameon=False, fontsize=8,
                  bbox_to_anchor=(0.5, y))


def draw_marginal(axis, real_histogram, synthetic_histogram, method, edges):
    style_axis(axis)
    for name, histogram, edge_color in [("真实数据", real_histogram, "#555555"),
                                       (method, synthetic_histogram, plot_color(method))]:
        axis.stairs(histogram[0], edges, fill=True, linewidth=0.8,
                    edgecolor=edge_color, facecolor=to_rgba(plot_color(name), 0.35))
    outside = synthetic_histogram[1]
    annotation = f"Out of [0, 1]: {outside:.1%}" if np.isfinite(outside) else "Out of [0, 1]: n/a"
    axis.text(0.98, 0.95, annotation, transform=axis.transAxes, fontsize=6.5, va="top", ha="right")
    axis.set(xlim=(0, 1), xticks=np.linspace(0, 1, 6))
    axis.yaxis.set_major_locator(MaxNLocator(3))
    axis.tick_params(labelsize=7, length=3)


def plot_marginal_regions(datasets, rows, methods, regions, output):
    selected = regions["原region索引"].to_numpy(dtype=int)
    regions.to_csv(output / "A1_plot_regions.csv", index=False)
    values = {name: dense_block(datasets[name], rows[name], selected) for name in ["真实数据", *methods]}
    edges = np.linspace(0, 1, 31)
    figure, axes = plt.subplots(len(selected), len(methods), figsize=(3.15 * len(methods), 2.0 * len(selected) + 0.7),
                                sharex=True, sharey="row", squeeze=False)
    figure.subplots_adjust(left=0.075, right=0.995, bottom=0.09, top=0.90, wspace=0.10, hspace=0.34)
    source_tables = []
    for i, row in regions.iterrows():
        histograms = {name: probability_histogram(values[name][:, i], edges) for name in ["真实数据", *methods]}
        upper = max(0.1, np.ceil(max(item[0].max() for item in histograms.values()) * 12) / 10)
        for name, (probabilities, outside, count) in histograms.items():
            source_tables.append(pd.DataFrame({"原region": row["原region"], "方法": name,
                "分箱左界": edges[:-1], "分箱右界": edges[1:], "概率": probabilities,
                "有限值数": count, "越界比例": outside}))
        for j, method in enumerate(methods):
            axis = axes[i, j]
            draw_marginal(axis, histograms["真实数据"], histograms[method], method, edges)
            axis.text(0.02, 0.95, str(row["原region"]), transform=axis.transAxes, fontsize=6, va="top", color="#555555")
            axis.set_ylim(0, upper)
            if i == 0:
                axis.set_title(plot_name(method), color=plot_color(method), fontsize=10, weight="bold", pad=7)
            if i == len(selected) - 1:
                axis.set_xlabel("Methylation level", fontsize=8)
            if j == 0:
                axis.set_ylabel(f"{row['展示名称']}\nProbability per bin", fontsize=8)
    marginal_legend(figure, methods, 0.98)
    figure.suptitle("Literature-guided Tle4 locus windows (mm10)" if regions["展示名称"].str.startswith("Tle4").all()
                     else "Literature-guided marginal distributions", fontsize=11, weight="bold", y=1.01)
    figure.text(0.075, 0.015, "All cell types pooled. Shared 30 bins; finite values only; out-of-range mass is not clipped.\n"
                "Windows were selected using real training cells within the published example locus; full-genome scores are unchanged.",
                fontsize=7, color="#555555")
    pd.concat(source_tables, ignore_index=True).to_csv(output / "A1_plot_histograms.csv", index=False)
    save_figure(figure, output, "A1_marginal_regions")


def plot_region_by_type(datasets, rows, methods, regions, output):
    detail = regions.loc[regions["单区域展示"].eq(1)].iloc[0]
    levels = detail["展示细胞类型"].split("|")
    names = ["真实数据", *methods]
    labels = {name: read_labels(datasets[name], rows[name]) for name in names}
    values = {name: dense_block(datasets[name], rows[name], [int(detail["原region索引"])]).ravel() for name in names}
    for name in names:
        if any(not np.any(labels[name] == level) for level in levels):
            raise ValueError(f"{name} 缺少清单指定的展示细胞类型，不静默混合或替换标签")
    edges = np.linspace(0, 1, 31)
    histograms = {(name, level): probability_histogram(values[name][labels[name] == level], edges)
                  for name in names for level in levels}
    upper = max(0.1, np.ceil(max(item[0].max() for item in histograms.values()) * 12) / 10)
    figure, axes = plt.subplots(len(methods), len(levels), figsize=(3.0 * len(levels), 1.9 * len(methods) + 1),
                                sharex=True, sharey=True, squeeze=False)
    figure.subplots_adjust(left=0.09, right=0.995, bottom=0.09, top=0.86, wspace=0.12, hspace=0.24)
    for i, method in enumerate(methods):
        for j, level in enumerate(levels):
            axis = axes[i, j]
            real_histogram, synthetic_histogram = histograms["真实数据", level], histograms[method, level]
            draw_marginal(axis, real_histogram, synthetic_histogram, method, edges)
            axis.text(0.02, 0.95, f"n: R={real_histogram[2]}, S={synthetic_histogram[2]}",
                      transform=axis.transAxes, fontsize=6.5, va="top", color="#555555")
            axis.set_ylim(0, upper)
            if i == 0:
                axis.set_title(level.replace(" CTX Glut", "\nCTX Glut"), fontsize=9, pad=8)
            if j == 0:
                axis.set_ylabel(f"{plot_name(method)}\nProbability per bin", color=plot_color(method), fontsize=8)
            if i == len(methods) - 1:
                axis.set_xlabel("Methylation level", fontsize=8)
    marginal_legend(figure, methods, 0.95)
    figure.suptitle(f"{detail['展示名称']} | {detail['原region']}\nSingle-region distributions by Subclass",
                   fontsize=11, weight="bold", y=1.01)
    figure.text(0.09, 0.015, "Rows: methods. Columns: saved cell types. R = evaluation real; S = synthetic; n counts finite observations.\n"
                "Shared 30 bins and y-axis; out-of-range values are reported separately.", fontsize=7, color="#555555")
    source_tables = [pd.DataFrame({"原region": detail["原region"], "细胞类型": level, "方法": name,
        "分箱左界": edges[:-1], "分箱右界": edges[1:], "概率": probability,
        "有限值数": count, "越界比例": outside})
        for (name, level), (probability, outside, count) in histograms.items()]
    pd.concat(source_tables, ignore_index=True).to_csv(output / "A1_type_histograms.csv", index=False)
    save_figure(figure, output, "A1_region_by_type")


def plot_a1(datasets, rows, methods, regions, output):
    plot_marginal_regions(datasets, rows, methods, regions, output)
    plot_region_by_type(datasets, rows, methods, regions, output)


def draw_correlation_violin(axis, vector, position, color, seed):
    finite = np.asarray(vector)[np.isfinite(vector)]
    if not finite.size:
        axis.text(position, 0, "No valid pairs", rotation=90, ha="center", fontsize=8)
        return
    median = np.median(finite)
    values = np.random.default_rng(seed).choice(finite, 200000, replace=False) if finite.size > 200000 else finite
    if values.size > 1 and np.ptp(values) > 1e-12:
        violin = axis.violinplot(values, positions=[position], widths=0.8, showextrema=False, points=200)
        for body in violin["bodies"]:
            body.set(facecolor=color, edgecolor="none", alpha=0.72)
        axis.hlines(median, position - 0.13, position + 0.13,
                    color="white" if color == "#D95F02" else "#E8793E", linewidth=1)
    else:
        axis.hlines(median, position - 0.25, position + 0.25, color=color, linewidth=1.4)


def plot_duplicate_pearson(correlations, summary, output):
    """小提琴展示有效对分布；柱高直接读取正式汇总，避免重画改变得分。"""
    methods = summary.loc[~summary["方法"].eq("真实数据"), "方法"].tolist()
    table = summary.set_index("方法").loc[methods]
    pair_count = int(summary["配对数"].max())
    figure, axes = plt.subplots(1, 2, figsize=(12.8, 4.7))
    figure.subplots_adjust(left=0.06, right=0.995, bottom=0.26, top=0.84, wspace=0.12)
    figure.suptitle(f"Dependency preservation after appending {pair_count:,} duplicated regions", fontsize=12, weight="bold", y=0.98)
    for position, method in enumerate(["真实数据", *methods], 1):
        vector = correlations.loc[correlations["方法"].eq(method), "Pearson"].to_numpy()
        draw_correlation_violin(axes[0], vector, position, plot_color(method), 42)
    axes[0].axhline(1, color="#888888", linestyle="--", linewidth=0.8, zorder=0)
    axes[0].set(title="Original vs duplicated regions", ylabel="Pearson correlation", ylim=(-1.05, 1.05),
                xticks=np.arange(1, len(methods) + 2), xticklabels=[plot_name(name) for name in ["真实数据", *methods]])
    means = table["Pearson均值"].to_numpy(dtype=float)
    axes[1].bar(np.arange(len(methods)), means, color=[plot_color(name) for name in methods], width=0.8)
    finite = means[np.isfinite(means)]
    lower = min(-0.05, float(finite.min()) * 1.25) if len(finite) else -0.05
    axes[1].set(title="Mean Pearson across valid region pairs", ylabel="Mean Pearson (higher is better)",
                ylim=(lower, 1.12), xticks=np.arange(len(methods)), xticklabels=[plot_name(name) for name in methods])
    for position, value in enumerate(means):
        label = f"{value:.4f}" if np.isfinite(value) else "NA"
        y = value + (0.02 if value >= 0 else -0.02) if np.isfinite(value) else 0.02
        axes[1].text(position, y, label, ha="center", va="bottom" if not np.isfinite(value) or value >= 0 else "top", fontsize=8)
    for axis in axes:
        style_axis(axis)
        axis.title.set_fontsize(10)
        axis.tick_params(axis="x", labelrotation=30, labelsize=8)
        for label in axis.get_xticklabels():
            label.set_ha("right")
    counts = "; ".join(f"{plot_name(name)} {int(table.loc[name, '有效配对数']):,}/{int(table.loc[name, '配对数']):,}" for name in methods)
    figure.text(0.06, 0.015, "Valid pairs: " + counts, fontsize=7, color="#555555")
    save_figure(figure, output, "A2_duplicate_Pearson")


def plot_joint_umap(coordinates, summary, methods, output, seed):
    """可直接用本轮正式坐标重绘；图上miLISI来自正式PCA空间得分。"""
    figure, axes = plt.subplots(1, len(methods), figsize=(4 * len(methods), 4.4), squeeze=False)
    figure.subplots_adjust(left=0.035, right=0.995, bottom=0.16, top=0.80, wspace=0.15)
    scores = summary.set_index("方法")["miLISI"]
    for axis, method in zip(axes.ravel(), methods):
        local = coordinates.loc[coordinates["方法"].eq(method)]
        xy = local[["UMAP1", "UMAP2"]].to_numpy()
        source = local["来源"].to_numpy()
        draw = np.random.default_rng(seed).permutation(len(local))
        for name, label, color in [("真实", "Real", "#A6A6A6"), ("合成", "Synthetic", plot_color(method))]:
            keep = draw[source[draw] == name]
            axis.scatter(xy[keep, 0], xy[keep, 1], s=8, alpha=0.65, color=color, linewidths=0, label=label, rasterized=True)
        axis.set(title=f"{plot_name(method)}\nmiLISI = {scores.loc[method]:.4f}", xlabel="UMAP 1", ylabel="UMAP 2", xticks=[], yticks=[])
        axis.title.set_color(plot_color(method))
        axis.title.set_fontsize(10)
        axis.legend(frameon=False, fontsize=8, markerscale=2)
        axis.spines[["top", "right"]].set_visible(False)
    figure.suptitle(f"Real / synthetic mixing (k = {int(summary['近邻数'].iloc[0])})", fontsize=12, weight="bold", y=0.98)
    figure.text(0.035, 0.025, "Each panel has its own joint UMAP in real-reference PCA space. Coordinates are not comparable across panels.\n"
                "miLISI is computed in PCA space, not from these 2D projections.", fontsize=7, color="#555555")
    save_figure(figure, output, "A3_joint_UMAP")


def run_a3(datasets, rows, methods, fill, scores, output, config):
    selected = select_features(scores, config["a3_top_regions"])
    n_components = min(config["pca_components"], len(rows["训练参照"]) - 1, len(selected))
    if n_components < 2 or len(rows["训练参照"]) < 4:
        raise ValueError("真实参照不足以拟合至少二维的 PCA 和 UMAP")
    pd.DataFrame({"原region索引": selected, "原region": datasets["训练参照"].var_names[selected],
                  "ANOVA": scores[selected], "填补中位数": fill[selected]}).to_csv(output / "A3_selected_regions.csv", index=False)
    np.save(output / "reference_natural_medians.npy", fill)
    pca = PCA(n_components=n_components, svd_solver="randomized", random_state=config["seed"])
    representations = {"训练参照": pca.fit_transform(imputed_features(
        datasets["训练参照"], rows["训练参照"], selected, fill))}
    names = ["训练参照", "真实数据", *methods]
    labels = {name: read_labels(datasets[name], rows[name]) for name in names}
    for name in names[1:]:
        if datasets[name] is datasets["训练参照"] and np.array_equal(rows[name], rows["训练参照"]):
            representations[name] = representations["训练参照"]
        else:
            representations[name] = pca.transform(imputed_features(datasets[name], rows[name], selected, fill))
    pc_tables = []
    for name in names:
        frame = pd.DataFrame(representations[name], columns=[f"PC{i + 1}" for i in range(n_components)])
        frame.insert(0, "细胞类型", labels[name])
        frame.insert(0, "细胞ID", datasets[name].obs_names[rows[name]])
        frame.insert(0, "数据集", name)
        pc_tables.append(frame)
    pd.concat(pc_tables, ignore_index=True).to_csv(output / "A3_PCA_coordinates.csv", index=False)
    summaries, cell_tables, group_tables = [], [], []
    for method in methods:
        score, cells, groups = milisi(representations["真实数据"], labels["真实数据"],
            representations[method], labels[method], config["milisi_neighbors"])
        cells.insert(0, "细胞ID", np.r_[datasets["真实数据"].obs_names[rows["真实数据"]],
                                      datasets[method].obs_names[rows[method]]])
        cells.insert(0, "方法", method)
        groups.insert(0, "方法", method)
        cell_tables.append(cells)
        group_tables.append(groups)
        summaries.append({"方法": method, "miLISI": score, "近邻数": config["milisi_neighbors"],
                          "入选区域数": len(selected), "PCA维数": n_components})
    summary = pd.DataFrame(summaries)
    summary.to_csv(output / "A3_miLISI.csv", index=False)
    pd.concat(cell_tables, ignore_index=True).to_csv(output / "A3_miLISI_cells.csv", index=False)
    pd.concat(group_tables, ignore_index=True).to_csv(output / "A3_type_details.csv", index=False)
    print("A3 的 PCA 和 miLISI 已计算，开始绘制 UMAP。", flush=True)
    parameters = dict(n_neighbors=min(config["milisi_neighbors"], len(rows["训练参照"]) - 1),
                      random_state=config["umap_seed"], transform_seed=config["umap_seed"], n_jobs=1)
    joint_tables = []
    for method in methods:
        combined = np.vstack([representations["真实数据"], representations[method]])
        joint_parameters = {**parameters, "n_neighbors": min(config["milisi_neighbors"], len(combined) - 1)}
        xy = umap.UMAP(**joint_parameters).fit_transform(combined)
        source = np.r_[np.repeat("Real", len(rows["真实数据"])), np.repeat("Synthetic", len(rows[method]))]
        joint_tables.append(pd.DataFrame({"方法": method, "来源": np.where(source == "Real", "真实", "合成"),
            "细胞ID": np.r_[datasets["真实数据"].obs_names[rows["真实数据"]], datasets[method].obs_names[rows[method]]],
            "UMAP1": xy[:, 0], "UMAP2": xy[:, 1]}))
    coordinates = pd.concat(joint_tables, ignore_index=True)
    coordinates.to_csv(output / "A3_joint_coordinates.csv", index=False)
    plot_joint_umap(coordinates, summary, methods, output, config["umap_seed"])
    return summary, {"入选区域数": len(selected), "PCA维数": n_components,
                     "解释方差比合计": float(pca.explained_variance_ratio_.sum())}

## 完整评估

每种合成方法与评估真实细胞等量，固定种子无放回抽样，A1/A2/A3 复用同一批细胞。
若某种合成方法缺少真实类型，A1 的相关加权指标保留缺失，不重新分配类型权重；完整缺少标签字段则停止并指出原因。

提供独立 reference_h5ad 时，要求与 real_h5ad 没有细胞 ID 重叠，参照预处理只在前者拟合。不提供时使用真实评估数据拟合，并在运行记录中明确标记。

In [ ]:
def evaluate_all(config):
    methods = list(config["synthetic_h5ad"])
    paths = {"真实数据": resolve_path(config["real_h5ad"]),
             **{name: resolve_path(value) for name, value in config["synthetic_h5ad"].items()}}
    reference_path = resolve_path(config["reference_h5ad"]) if config["reference_h5ad"] else paths["真实数据"]
    pair_path = resolve_path(config["region_pairs_csv"]) if config["region_pairs_csv"] else None
    region_path = resolve_path(config["a1_regions_csv"]) if config.get("a1_regions_csv") else None
    output = resolve_path(config["output_dir"])
    if sys.platform != "win32":
        output.relative_to(Path("/home/sccasimp/data/xb/outputs/simscDNAm").resolve())
    all_paths = set(paths.values()) | {reference_path}
    if pair_path:
        all_paths.add(pair_path)
    if region_path:
        all_paths.add(region_path)
    if any(path.is_relative_to(output) for path in all_paths):
        raise ValueError("输出目录不能包含输入文件，以免覆盖输入；请选择独立的评估输出目录")
    for path in all_paths:
        if not path.is_file():
            raise FileNotFoundError(f"缺少输入文件：{path}")
    if len(set(paths.values())) != len(paths):
        raise ValueError("真实数据和各方法合成数据必须指向不同文件")
    records = [{"路径": str(path), "字节数": path.stat().st_size, "修改时间纳秒": path.stat().st_mtime_ns}
               for path in sorted(all_paths)]
    output.mkdir(parents=True, exist_ok=True)
    (output / "figures").mkdir(exist_ok=True)
    report = {"状态": "运行中", "标签字段": "Subclass", "最后复制区间数": 1000, "方法": methods,
              "参照模式": "独立训练真实参照" if reference_path != paths["真实数据"] else "评估真实数据拟合参照，非独立留出评估",
              "配置": config, "配置路径": str(CONFIG_PATH), "输入文件": records, "内核": sys.executable,
              "依赖版本": {name: importlib.metadata.version(name) for name in
                  ["numpy", "pandas", "scipy", "anndata", "scikit-learn", "umap-learn"]}}
    def write_report():
        (output / "run_summary.json").write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
    write_report()
    try:
        with ExitStack() as stack, threadpool_limits(limits=8):
            datasets = {}
            for name, path in paths.items():
                item = ad.read_h5ad(path, backed=None if config.get("read_in_memory", False) else "r")
                stack.callback(item.file.close)
                datasets[name] = item
            real = datasets["真实数据"]
            if reference_path == paths["真实数据"]:
                datasets["训练参照"] = real
            else:
                item = ad.read_h5ad(reference_path, backed=None if config.get("read_in_memory", False) else "r")
                stack.callback(item.file.close)
                datasets["训练参照"] = item
                if len(real.obs_names.intersection(item.obs_names)):
                    raise ValueError("独立训练参照与评估真实细胞 ID 重叠，请提供分离后的数据")
            for name, item in datasets.items():
                validate_region_axis(real, item)
                if not item.obs_names.is_unique:
                    raise ValueError(f"{name} 的细胞 ID 不唯一")
            pairs = region_pairs(real, pair_path=pair_path)
            validate_real_duplicates(real, pairs)
            if datasets["训练参照"] is not real:
                validate_real_duplicates(datasets["训练参照"], pairs)
            natural_count = real.n_vars - 1000
            if region_path is None:
                raise ValueError("请通过 a1_regions_csv 提供有文献依据的展示清单；不再随机选择区域")
            plot_regions = load_plot_regions(real, natural_count, region_path, config.get("genome_assembly"))
            count = real.n_obs if config["evaluation_cells"] is None else config["evaluation_cells"]
            rows = {name: sample_rows(item, count, config["seed"]) for name, item in datasets.items() if name != "训练参照"}
            rows["训练参照"] = np.arange(datasets["训练参照"].n_obs)
            labels = {name: read_labels(item, rows[name]) for name, item in datasets.items()}
            if 2 * count <= config["milisi_neighbors"]:
                raise ValueError("合并细胞数必须大于 miLISI 近邻数")
            pairs.to_csv(output / "region_pairs.csv", index=False)
            pd.concat([pd.DataFrame({"数据集": name, "原行索引": selected,
                "细胞ID": datasets[name].obs_names[selected], "细胞类型": labels[name]})
                for name, selected in rows.items()], ignore_index=True).to_csv(output / "selected_cells.csv", index=False)
            report.update({"评估细胞数": count, "真实参照细胞数": len(rows["训练参照"]), "原始区间数": natural_count,
                           "总区间数": real.n_vars, "A1_A3排除末尾复制列": True})
            write_report()
            print(f"输入校验通过：原始区间 {natural_count}，复制区间 1000，每种方法评估 {count} 个细胞。", flush=True)

            print("开始 A1：按真实类型加权的边缘指标。", flush=True)
            a1_rows, detail_tables = [], []
            for method in methods:
                summary, details = a1_metrics(real, datasets[method], rows["真实数据"], rows[method], natural_count)
                a1_rows.append({"方法": method, **summary})
                details.insert(0, "方法", method)
                detail_tables.append(details)
            a1 = pd.DataFrame(a1_rows)
            details = pd.concat(detail_tables, ignore_index=True)
            a1.to_csv(output / "A1_summary.csv", index=False)
            details.to_csv(output / "A1_type_details.csv", index=False)
            plot_a1(datasets, rows, methods, plot_regions, output)

            print("开始 A2：读取实际复制列，计算 1000 对 Pearson。", flush=True)
            pair_tables = []
            for name in ["真实数据", *methods]:
                original = dense_block(datasets[name], rows[name], pairs["原region索引"].to_numpy())
                copied = dense_block(datasets[name], rows[name], pairs["复制region索引"].to_numpy())
                frame = pd.concat([pairs, paired_pearson(original, copied)], axis=1)
                frame.insert(0, "方法", name)
                pair_tables.append(frame)
            correlations = pd.concat(pair_tables, ignore_index=True)
            a2 = correlations.groupby("方法", sort=False).agg(配对数=("Pearson", "size"),
                有效配对数=("Pearson", "count"), Pearson均值=("Pearson", "mean"), Pearson中位数=("Pearson", "median")).reset_index()
            a2["无效配对数"] = a2["配对数"] - a2["有效配对数"]
            correlations.to_csv(output / "A2_duplicate_correlations.csv", index=False)
            a2.to_csv(output / "A2_pearson_summary.csv", index=False)
            plot_duplicate_pearson(correlations, a2, output)

            print("仅用真实参照的原始区间拟合中位数、按 Subclass 的 ANOVA。", flush=True)
            fill, scores = natural_reference_statistics(datasets["训练参照"], rows["训练参照"], natural_count)
            selected = select_features(scores, config["spearman_top_regions"])
            pd.DataFrame({"原region索引": selected, "原region": real.var_names[selected], "ANOVA": scores[selected]}).to_csv(
                output / "A2_natural_selected_regions.csv", index=False)
            spearman_rows = []
            for index, name in enumerate(["真实数据", *methods]):
                vector = spearman_vector(imputed_features(datasets[name], rows[name], selected, fill))
                filename = f"A2_Spearman_{index:02d}.npy"
                np.save(output / filename, vector)
                finite = vector[np.isfinite(vector)]
                spearman_rows.append({"方法": name, "入选原始区域数": len(selected), "区域对数": len(vector),
                    "有效区域对数": len(finite), "Spearman均值": finite.mean() if len(finite) else np.nan,
                    "Spearman中位数": np.median(finite) if len(finite) else np.nan, "向量文件": filename})
            pd.DataFrame(spearman_rows).to_csv(output / "A2_natural_Spearman_summary.csv", index=False)

            print("开始 A3：按 Subclass 筛选原始区域，真实参照 PCA、miLISI 和 UMAP。", flush=True)
            a3, a3_protocol = run_a3(datasets, rows, methods, fill, scores, output, config)
            comparison = a1.merge(a2[["方法", "Pearson均值", "有效配对数"]].rename(columns={"Pearson均值": "A2_Pearson均值"}), on="方法").merge(a3, on="方法")
            comparison.to_csv(output / "A1_A2_A3_summary.csv", index=False)
            diagnostics = []
            for name in ["真实数据", *methods]:
                for domain, start, stop in [("原始区间", 0, natural_count), ("复制区间", natural_count, real.n_vars)]:
                    valid, outside = 0, 0
                    for left in range(start, stop, 512):
                        block = dense_block(datasets[name], rows[name], slice(left, min(left + 512, stop)))
                        finite = np.isfinite(block)
                        valid += int(finite.sum())
                        outside += int((finite & ((block < 0) | (block > 1))).sum())
                    diagnostics.append({"数据集": name, "区间范围": domain, "有限观测数": valid,
                        "缺失比例": 1 - valid / (len(rows[name]) * (stop - start)),
                        "超出0到1比例": outside / valid if valid else np.nan})
            pd.DataFrame(diagnostics).to_csv(output / "input_diagnostics.csv", index=False)
            for record in records:
                path = Path(record["路径"])
                if path.stat().st_size != record["字节数"] or path.stat().st_mtime_ns != record["修改时间纳秒"]:
                    raise RuntimeError(f"评估期间输入文件发生变化：{path}")
            report.update({"状态": "完成", "输入文件大小及修改时间未变": True, "A3": a3_protocol,
                           "输出目录": str(output), "输出文件": sorted(str(p.relative_to(output)) for p in output.rglob("*") if p.is_file())})
            write_report()
            display(comparison)
            print("统一评估完成，结果目录：", output)
            return comparison
    except Exception as error:
        report.update({"状态": "失败", "错误": f"{type(error).__name__}: {error}"})
        write_report()
        raise


comparison = evaluate_all(config)

## 解释结果

A1 数值评分仍覆盖全部原始区间；两张边缘分布图是有文献依据的案例展示，不能替代全基因组指标。真实评估细胞和合成细胞使用同一 Subclass 标签分组，不进行填补、截断或按模型表现换区间。

A2 小提琴呈现有效复制对的分布；柱高直接来自正式 Pearson 汇总。自然区间 Spearman 仍保存数值与完整向量，不绘辅助图。

A3 仅绘逐方法真实／合成混合图。各面板单独拟合联合 UMAP，坐标不可跨面板直接比较；miLISI 在真实参照 PCA 空间计算，不能从二维重叠程度直接读出。
